# Lesson 09 · The multinomial and the Dirichlet

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 10 (pp. 89–98)

**Question:** Not all hits are equal: a home run is worth more than a single. Who were the best sluggers in history once we account for small samples?

**Goal:** Move from two outcomes to five (single, double, triple, home run, out), using the multinomial and its conjugate prior, the Dirichlet. Fit a Dirichlet-multinomial prior and shrink slugging percentages.

**Contents**

1. Setup: hit types
2. Slugging percentage, raw
3. The multinomial and the Dirichlet, by simulation
4. Fitting a Dirichlet-multinomial prior
5. Checking the fit
6. Shrinking slugging percentage
7. Exercises

**Tags:** `dirichlet` `multinomial` `dirichlet-multinomial` `slugging-percentage` `pseudo-counts` `maximum-likelihood`

**Before you start:** Lessons 01–02.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

hits = eb_data.hit_types(pitcher_rule=3)
categories = ["Single", "Double", "Triple", "HR", "NonHit"]
print(f"{len(hits):,} players")
hits.head()

## 1. Setup: hit types

**Your turn.** Confirm that `Single + Double + Triple + HR + NonHit == AB` for every player. Show Hank Aaron's row.

In [ ]:
# Your code here

> **Check:** Hank Aaron: 12,364 AB, 2,294 singles, 624 doubles, 98 triples, 755 HR.

## 2. Slugging percentage, raw

$$\text{SLG} = \frac{\text{Singles} + 2\cdot\text{Doubles} + 3\cdot\text{Triples} + 4\cdot\text{HR}}{AB},$$

which ranges from 0 to 4.

**Your turn.** Compute `slugging` and show the top 8. Then, for players with AB ≥ 500, plot histograms of each hit type as a fraction of AB.

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Who tops the raw slugging list, and why is that the same problem as in Lesson 02?

*Your answer:*

## 3. The multinomial and the Dirichlet, by simulation

Each at-bat is a roll of a five-sided die with probabilities $p_1, \dots, p_5$: a **multinomial**. Its conjugate prior is the **Dirichlet**($\alpha_1, \dots, \alpha_5$):
- the relative sizes $\alpha_k / \sum\alpha$ set where the probability mass goes;
- the total $\sum\alpha$ sets how tightly it concentrates there.

The beta is the two-category special case.

**Your turn.** Simulate a few multinomial draws with `rng.multinomial(100, [.2] * 5, size=3)` and `rng.multinomial(100, [.05, .05, .2, .2, .5], size=3)`.

In [ ]:
# Your code here

**Your turn.** Simulate 10,000 Dirichlet draws for (1, 1, 1, 1, 1), (5, 2, 2, 1, 1) and (500, 200, 200, 100, 100) with `rng.dirichlet`. Plot a histogram per category, one row per parameter set (a 3×5 grid).

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** (5, 2, 2, 1, 1) and (500, 200, 200, 100, 100) have the same means. What's different, and what does the total represent if you think of the α's as pseudo-counts?

*Your answer:*

## 4. Fitting a Dirichlet-multinomial prior

As in Lesson 02, average over the unknown probabilities. The result is the **Dirichlet-multinomial**, with log-pmf

$$\log P(x \mid \alpha) = \log\Gamma(A) + \log\Gamma(n+1) - \log\Gamma(n + A) + \sum_k \left[\log\Gamma(x_k + \alpha_k) - \log\Gamma(\alpha_k) - \log\Gamma(x_k + 1)\right],$$

where $A = \sum_k \alpha_k$ and $n = \sum_k x_k$. SciPy has `stats.dirichlet_multinomial`, but writing the formula with `special.gammaln` is quick, vectorizes over players, and makes the fit fast.

**Your turn.** Write `dm_neg_log_lik(log_alpha, X)` for a players × 5 count matrix `X`, then fit the players with AB ≥ 500 (optimize over log α). Check your log-pmf against `stats.dirichlet_multinomial.logpmf` for one player.

In [ ]:
# Your code here

In [ ]:
# Your code here

> **Check:** α ≈ Single 35.3, Double 8.42, Triple 1.85, HR 2.84, NonHit 138.

## 5. Checking the fit

Each category's marginal under the Dirichlet is a beta: $p_k \sim \text{Beta}(\alpha_k, A - \alpha_k)$.

**Your turn.** Overlay each category's fitted beta density on its histogram from Section 2 (AB ≥ 500).

In [ ]:
# Your code here

**Reflect.** Which categories does the Dirichlet fit well, and which badly? A Dirichlet has a single total controlling the spread of every category. Why is that a strong assumption?

*Your answer:*

## 6. Shrinking slugging percentage

Shrinkage is the same pseudo-count trick: add each $\alpha_k$ to its category's count, then compute slugging.

**Your turn.** Compute the prior mean slugging (weights 1, 2, 3, 4, 0 applied to α / A), then the shrunken `slugging_eb` for every player.

In [ ]:
# Your code here

> **Check:** prior mean slugging ≈ 0.370.

**Your turn.** Plot raw against shrunken slugging, coloured by AB, and list the top 10 by `slugging_eb`.

In [ ]:
# Your code here

> **Check:** Ted Williams, Lou Gehrig and Barry Bonds should lead.

## 7. Exercises

**Your turn.** Shrink the home-run rate on its own with a beta-binomial (HR vs not-HR). Compare the fitted Beta with the Dirichlet's HR marginal. Should they agree? Do they?

In [ ]:
# Your code here

**Reflect.** **Card connection.** A booster pack's rare slot can land on one of several rarities. If you've logged the rarities pulled from each set, how would a Dirichlet-multinomial help estimate each set's rarity mix? What do the pseudo-counts mean there?

*Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 10.
- Minka (2000), "Estimating a Dirichlet distribution" (technical report): fixed-point algorithms for the fit.
- SciPy `dirichlet_multinomial`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.dirichlet_multinomial.html
- Wikipedia, "Conjugate prior" (table of conjugate pairs): https://en.wikipedia.org/wiki/Conjugate_prior